### Ex3.1

1\.a Make a new directory called `students` in your home. Download a csv file with the list of students of this lab from [here](https://www.dropbox.com/s/867rtx3az6e9gm8/LCP_22-23_students.csv) (use the `wget` command) and copy that to `students`. First check whether the file is already there \

1\.b Make two new files, one containing the students belonging to PoD, the other to Physics. \

1\.c For each letter of the alphabet, count the number of students whose surname starts with that letter. \

1\.d Find out which is the letter with most counts. \

1\.e Assume an obvious numbering of the students in the file (first line is 1, second line is 2, etc.), group students "modulo 18", i.e. 1,19,37,.. 2,20,38,.. etc. and put each group in a separate file  

In [11]:
'''
%%bash
pwd
git branch --show-current
git status --shor
'''

'\n%%bash\npwd\ngit branch --show-current\ngit status --shor\n'

In [2]:
'''

%%writefile ex3_1_students.sh
#!/bin/bash

# Exercise 3.1 — Bash student list processing

# 1.a Create students directory in home
mkdir -p "$HOME/students"

# Go to the students directory
cd "$HOME/students" || exit

# Define file name and URL
file="LCP_22-23_students.csv"
url="https://www.dropbox.com/s/867rtx3az6e9gm8/LCP_22-23_students.csv"

# Check whether the file is already there
if [ -f "$file" ]; then
    echo "$file already exists."
else
    echo "$file not found. Downloading..."
    wget -O "$file" "$url"
fi

# 1.b Create two files: PoD students and Physics students
grep "PoD" "$file" > students_PoD.csv
grep "Physics" "$file" > students_Physics.csv

# 1.c Count how many surnames start with each letter
> surname_counts.txt

for letter in {A..Z}
do
    count=$(awk -F',' -v l="$letter" 'toupper($1) ~ "^"l {count++} END {print count+0}' "$file")
    echo "$letter $count" >> surname_counts.txt
done

# 1.d Find the letter with the maximum count
sort -k2 -nr surname_counts.txt | head -n 1 > most_common_letter.txt

# 1.e Group students modulo 18
rm -f group_*.txt

awk -F',' '
{
    group = ((NR - 1) % 18) + 1
    print $0 >> "group_" group ".txt"
}
' "$file"

echo "Exercise 3.1 completed."
'''

Writing ex3_1_students.sh


In [40]:
%%writefile ex3_1_students.sh
#!/bin/bash

set -e

# Exercise 3.1 — Final clean exam-style solution

# 1.a Create students directory in home
mkdir -p "$HOME/students"

# Define file and URL
file="$HOME/students/LCP_22-23_students.csv"
url="https://www.dropbox.com/s/867rtx3az6e9gm8/LCP_22-23_students.csv?dl=1"

# Download file only if it is not already there
if [ -f "$file" ]; then
    echo "CSV file already exists."
else
    echo "CSV file not found. Downloading..."

    if command -v wget >/dev/null 2>&1; then
        wget -O "$file" "$url"
    else
        curl -L -o "$file" "$url"
    fi
fi

# 1.b Create files for PoD and Physics students
awk -F',' 'NR > 1 && $4 == "PoD" {print $0}' "$file" > "$HOME/students/students_PoD.csv"
awk -F',' 'NR > 1 && $4 == "Physics" {print $0}' "$file" > "$HOME/students/students_Physics.csv"

# 1.c Count students whose surname starts with each letter
> "$HOME/students/surname_counts.txt"

for letter in {A..Z}
do
    awk -F',' -v letter="$letter" '
    NR > 1 {
        first_letter = toupper(substr($1, 1, 1))
        if (first_letter == letter) {
            count++
        }
    }
    END {
        print letter, count + 0
    }
    ' "$file" >> "$HOME/students/surname_counts.txt"
done

# 1.d Find the most common surname starting letter
sort -k2,2nr "$HOME/students/surname_counts.txt" | head -n 1 > "$HOME/students/most_common_letter.txt"

# 1.e Split students into 18 groups using modulo 18
awk -F',' '
NR > 1 {
    group = ((NR - 2) % 18) + 1
    print $0 > ENVIRON["HOME"] "/students/group_" group ".txt"
}
' "$file"

echo "Exercise 3.1 completed."

Overwriting ex3_1_students.sh


In [41]:
!bash -lc 'bash ex3_1_students.sh'

CSV file already exists.
Exercise 3.1 completed.


In [45]:
!bash -lc 'ls "$HOME/students"/group_*.txt | wc -l'

18


In [46]:
!bash -lc 'cat "$HOME/students/most_common_letter.txt"'

B 13


In [47]:
!bash -lc 'wc -l "$HOME/students/students_PoD.csv"'

67 /c/Users/sisto/students/students_PoD.csv


In [48]:
!bash -lc 'wc -l "$HOME/students/students_Physics.csv"'

5 /c/Users/sisto/students/students_Physics.csv


### Ex3.2
2.a Make a copy of the file `data.csv` removing the metadata and the commas between numbers; call it `data.txt` \
2\.b How many even numbers are there? \
2\.c Distinguish the entries on the basis of `sqrt(X^2 + Y^2 + Z^2)` is greater or smaller than `100*sqrt(3)/2`. Count the entries of each of the two groups \
2\.d Make `n` copies of data.txt (with `n` an input parameter of the script), where the i-th copy has all the numbers divided by i (with `1<=i<=n`).